In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Load the dataset
df_victim_details = pd.read_csv('/mnt/data/Updated_Dataset_with_Imputed_Values.csv')  # Path to your dataset

# Assuming 'Serial' is the target column (1 for serial, 0 for non-serial)
df_victim_details['Serial'] = df_victim_details['Incident'].apply(lambda x: 1 if x == 1 else 0)

# Select victim-related columns and the target
victim_columns = ['Victim Age', 'Victim Count', 'Victim Height', 'Victim Weight', 'Gender', 
                  'Ethnicity', 'Age Group', 'Psychological Traits', 'Behavioral Patterns']
X_victim = df_victim_details[victim_columns]
y = df_victim_details['Serial']

# Preprocess Victim Details
X_victim.fillna(X_victim.mean(), inplace=True)
scaler_victim = StandardScaler()
X_victim_scaled = scaler_victim.fit_transform(X_victim)

# Train Test Split for Victim Details
X_train_victim, X_test_victim, y_train, y_test = train_test_split(X_victim_scaled, y, test_size=0.2, random_state=42)

# Neural Network Model for Victim Details (Already trained in the previous step)
nn_victim_model = Sequential()
nn_victim_model.add(Dense(128, input_dim=X_train_victim.shape[1], activation='relu'))
nn_victim_model.add(Dense(256, activation='relu'))
nn_victim_model.add(Dense(128, activation='relu'))
nn_victim_model.add(Dense(64, activation='relu'))
nn_victim_model.add(Dense(32, activation='relu'))
nn_victim_model.add(Dense(16, activation='relu'))
nn_victim_model.add(Dense(1, activation='sigmoid'))
nn_victim_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
nn_victim_model.fit(X_train_victim, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Forensic Model (Assumed already trained)
# For simplicity, I'm going to assume it's trained similarly to the Victim Details model.
# You would need to replace it with your actual trained model for forensic data
X_forensic = df_victim_details[['Forensic Column1', 'Forensic Column2', 'Forensic Column3']]  # Replace with actual columns
X_forensic.fillna(X_forensic.mean(), inplace=True)
scaler_forensic = StandardScaler()
X_forensic_scaled = scaler_forensic.fit_transform(X_forensic)

# Train Test Split for Forensic
X_train_forensic, X_test_forensic, y_train_forensic, y_test_forensic = train_test_split(X_forensic_scaled, y, test_size=0.2, random_state=42)

# Placeholder NN Model for Forensic Reports (Replace with actual trained model)
nn_forensic_model = Sequential()
nn_forensic_model.add(Dense(128, input_dim=X_train_forensic.shape[1], activation='relu'))
nn_forensic_model.add(Dense(256, activation='relu'))
nn_forensic_model.add(Dense(128, activation='relu'))
nn_forensic_model.add(Dense(64, activation='relu'))
nn_forensic_model.add(Dense(32, activation='relu'))
nn_forensic_model.add(Dense(16, activation='relu'))
nn_forensic_model.add(Dense(1, activation='sigmoid'))
nn_forensic_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
nn_forensic_model.fit(X_train_forensic, y_train_forensic, epochs=50, batch_size=32, validation_split=0.2)

# Geographical Model (Assumed already trained)
# Replace with actual geographical data columns
X_geographical = df_victim_details[['Geographical Column1', 'Geographical Column2']]  # Replace with actual columns
X_geographical.fillna(X_geographical.mean(), inplace=True)
scaler_geographical = StandardScaler()
X_geographical_scaled = scaler_geographical.fit_transform(X_geographical)

# Train Test Split for Geographical
X_train_geographical, X_test_geographical, y_train_geographical, y_test_geographical = train_test_split(X_geographical_scaled, y, test_size=0.2, random_state=42)

# Placeholder NN Model for Geographical Reports (Replace with actual trained model)
nn_geographical_model = Sequential()
nn_geographical_model.add(Dense(128, input_dim=X_train_geographical.shape[1], activation='relu'))
nn_geographical_model.add(Dense(256, activation='relu'))
nn_geographical_model.add(Dense(128, activation='relu'))
nn_geographical_model.add(Dense(64, activation='relu'))
nn_geographical_model.add(Dense(32, activation='relu'))
nn_geographical_model.add(Dense(16, activation='relu'))
nn_geographical_model.add(Dense(1, activation='sigmoid'))
nn_geographical_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
nn_geographical_model.fit(X_train_geographical, y_train_geographical, epochs=50, batch_size=32, validation_split=0.2)

# --- MAIN MODEL ---

# Use predictions from the three models as new features for the main model
victim_predictions = nn_victim_model.predict(X_test_victim)
forensic_predictions = nn_forensic_model.predict(X_test_forensic)
geographical_predictions = nn_geographical_model.predict(X_test_geographical)

# Combine predictions to create a new dataset for the main model
main_model_input = np.hstack([victim_predictions, forensic_predictions, geographical_predictions])

# Train the Main Model (Random Forest or NN)
main_rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
main_rf_model.fit(main_model_input, y_test)

# Evaluate the Main Model
main_rf_accuracy = main_rf_model.score(main_model_input, y_test)
print(f"Main Model (Random Forest) Accuracy: {main_rf_accuracy:.4f}")

# --- Neural Network Model for Main Model ---
main_nn_model = Sequential()
main_nn_model.add(Dense(128, input_dim=main_model_input.shape[1], activation='relu'))
main_nn_model.add(Dense(256, activation='relu'))
main_nn_model.add(Dense(128, activation='relu'))
main_nn_model.add(Dense(64, activation='relu'))
main_nn_model.add(Dense(32, activation='relu'))
main_nn_model.add(Dense(1, activation='sigmoid'))

main_nn_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
main_nn_model.fit(main_model_input, y_test, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the Main NN Model
main_nn_loss, main_nn_accuracy = main_nn_model.evaluate(main_model_input, y_test)
print(f"Main Model (Neural Network) Accuracy: {main_nn_accuracy:.4f}")
